# 3. Keyword Embeddings (Sparse Retrieval)

**RAG Pipeline Series — Notebook 3**

In notebook 2 we turned `rag.pdf` into chapter-tagged chunks using `RecursiveCharacterTextSplitter`. Before reaching for dense neural embeddings (notebook 4), this notebook builds the classic **sparse** representations that predate them: **TF-IDF** and **BM25**.

Fittingly, `rag.pdf` itself covers this exact history in Chapter 2 ("Evolution of Retrieval") — it explains TF-IDF, BM25, and *why* keyword search eventually gives way to semantic embeddings. We'll pull real passages from that chapter into the search examples below, so the document is (once again) self-aware about the topic at hand.

In this notebook we will:
1. Rebuild the chapter-tagged chunks from notebook 2 (same `chunk_size=800`, `chunk_overlap=100` baseline).
2. Turn every chunk into a **TF-IDF** sparse vector and search with cosine similarity.
3. Build a **BM25** index over the same chunks and compare rankings.
4. Run both on a keyword-heavy query and a paraphrased query, to see where sparse ranking holds up and where it starts to slip — setting up the problem notebook 4 (vector embeddings) solves.

## Setup

In [1]:
%pip install -q -U langchain langchain-community pypdf scikit-learn rank_bm25 pandas

Note: you may need to restart the kernel to use updated packages.


## 1. Recap: loading and chunking `rag.pdf`

Same loading + chapter-aware chunking as notebook 2: load pages with `PyPDFLoader`, strip the repeated header/footer lines, split the full text into per-chapter spans, then run `RecursiveCharacterTextSplitter` *within* each chapter so no chunk crosses a chapter boundary. Each chunk keeps its `chapter_num` / `chapter_title` as metadata.

In [ ]:
# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
from rag_utils import maybe_colab_upload

maybe_colab_upload()

In [1]:
from rag_utils import resolve_pdf_path, load_clean_text

# Prefer the Colab upload location; fall back to this repo's dataset/ folder
# when running locally (rag-notebooks/ and dataset/ are sibling folders).
PDF_PATH = resolve_pdf_path()
pages, full_text = load_clean_text(PDF_PATH)

print(f"Loaded {len(pages)} pages from {PDF_PATH}")

Loaded 58 pages from ..\dataset\rag.pdf


In [2]:
from rag_utils import DEFAULT_CHUNK_OVERLAP, DEFAULT_CHUNK_SIZE, chunk_chapters, split_into_chapters

CHUNK_SIZE, CHUNK_OVERLAP = DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP
chapters = split_into_chapters(full_text)
chunks = chunk_chapters(chapters, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"{len(chapters)} chapters -> {len(chunks)} chapter-tagged chunks")
print(chunks[0].metadata)
print(chunks[0].page_content[:200])

15 chapters -> 181 chapter-tagged chunks
{'chapter_num': '01', 'chapter_title': 'Introduction to RAG'}
Retrieval-Augmented Generation (RAG) is one of the most transformative patterns in modern AI
engineering. It bridges the gap between the remarkable language capabilities of Large Language
Models (LLMs


## 2. TF-IDF: a sparse vector per chunk

**TF-IDF** (term frequency - inverse document frequency) weights a word by how often it appears in a chunk, discounted by how common it is across *all* chunks. Every chunk becomes a vector as long as the vocabulary — almost entirely zeros. This is exactly the "Vector Space / TF-IDF" row of the `Table 2.1: Key milestones in information retrieval` that `rag.pdf`'s own Chapter 2 lays out.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [d.page_content for d in chunks]
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Chunk vector matrix shape: {tfidf_matrix.shape}")

row0 = tfidf_matrix[0].toarray()[0]
nonzero = (row0 > 0).sum()
print(f"\nChunk 0: {nonzero} nonzero entries out of {len(row0)} ({nonzero / len(row0):.1%}) -- this is what sparse means")

top_terms = sorted(vectorizer.vocabulary_.items(), key=lambda kv: -row0[kv[1]])[:8]
print("Top-weighted terms in chunk 0:", [t for t, _ in top_terms])

Vocabulary size: 2600
Chunk vector matrix shape: (181, 2600)

Chunk 0: 57 nonzero entries out of 2600 (2.2%) -- this is what sparse means
Top-weighted terms in chunk 0: ['language', 'models', 'large', 'transformative', 'bridges', 'growing', 'establishes', 'llama']


## 3. Searching with TF-IDF: cosine similarity between sparse vectors

To search, transform the query into the same sparse vector space and rank chunks by cosine similarity to the query vector.

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_search(query, k=5):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix)[0]
    order = sims.argsort()[::-1][:k]
    return [(int(i), round(float(sims[i]), 3), chunks[i].metadata["chapter_title"]) for i in order]

## 4. A keyword-style query vs. a paraphrased query

Chapter 2 of `rag.pdf` explains **BM25** in one chunk (mentioning "Okapi Best Match 25", "term frequency saturation", "document length normalization") and, a little earlier in Chapter 1, explains how RAG fixes **hallucination** by giving the model "a dynamic, external knowledge source" to consult. We locate both chunks by their exact wording, then query for the same ideas two different ways:

- A **keyword-style** query that reuses the chunk's own vocabulary almost verbatim.
- A **paraphrased** query that describes the same idea in different words, the way a real user actually types.

Sparse methods can only match words that are literally present, so watch how the correct chunk's rank changes between the two.

In [5]:
bm25_chunk_idx = next(i for i, d in enumerate(chunks) if "Okapi Best Match 25" in d.page_content)
hallucination_chunk_idx = next(i for i, d in enumerate(chunks) if "dynamic, external knowledge source" in d.page_content)

keyword_query = "Okapi Best Match 25 term frequency saturation document length normalization"
paraphrase_query = "How can giving a language model outside documents stop it from making things up?"

print("Keyword-style query:", keyword_query)
print("Target chunk:", bm25_chunk_idx, "-", chunks[bm25_chunk_idx].metadata)
print("TF-IDF top-5:", tfidf_search(keyword_query))

Keyword-style query: Okapi Best Match 25 term frequency saturation document length normalization
Target chunk: 18 - {'chapter_num': '02', 'chapter_title': 'Evolution of Retrieval'}
TF-IDF top-5: [(18, 0.464, 'Evolution of Retrieval'), (66, 0.222, 'Retrieval Techniques'), (16, 0.208, 'Evolution of Retrieval'), (49, 0.093, 'Embeddings'), (116, 0.086, 'Generation')]


In [6]:
print("Paraphrased query:", paraphrase_query)
print("Target chunk:", hallucination_chunk_idx, "-", chunks[hallucination_chunk_idx].metadata)
print("TF-IDF top-5:", tfidf_search(paraphrase_query))

Paraphrased query: How can giving a language model outside documents stop it from making things up?
Target chunk: 1 - {'chapter_num': '01', 'chapter_title': 'Introduction to RAG'}
TF-IDF top-5: [(112, 0.146, 'Generation'), (139, 0.124, 'RAG vs Fine-tuning'), (103, 0.12, 'Augmentation'), (1, 0.111, 'Introduction to RAG'), (0, 0.108, 'Introduction to RAG')]


Notice how much weaker the paraphrased query's ranking is compared to the keyword query's — TF-IDF has no notion that "outside documents" means the same thing as "external knowledge source". It can only match words that literally co-occur.

## 5. BM25: a stronger keyword ranking function

**BM25** improves on raw TF-IDF with two ideas Chapter 2 calls out directly: term-frequency **saturation** (the 100th occurrence of a word shouldn't count 100x as much) and **document-length normalization** (a long chunk isn't unfairly favored just for containing more words). It's still purely lexical — no semantic understanding — but it's the standard "keyword search" baseline used in real systems like Elasticsearch, and the first-stage retriever in many production RAG pipelines.

In [7]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [text.lower().split() for text in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_search(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    order = scores.argsort()[::-1][:k]
    return [(int(i), round(float(scores[i]), 2), chunks[i].metadata["chapter_title"]) for i in order]

print("Keyword-style query:", keyword_query)
print("BM25 top-5:", bm25_search(keyword_query))
print()
print("Paraphrased query:", paraphrase_query)
print("BM25 top-5:", bm25_search(paraphrase_query))

Keyword-style query: Okapi Best Match 25 term frequency saturation document length normalization
BM25 top-5: [(18, 32.98, 'Evolution of Retrieval'), (66, 11.06, 'Retrieval Techniques'), (16, 10.3, 'Evolution of Retrieval'), (20, 6.59, 'Evolution of Retrieval'), (116, 5.83, 'Generation')]

Paraphrased query: How can giving a language model outside documents stop it from making things up?
BM25 top-5: [(112, 13.82, 'Generation'), (6, 11.05, 'Introduction to RAG'), (1, 10.6, 'Introduction to RAG'), (11, 10.53, 'Introduction to RAG'), (139, 10.19, 'RAG vs Fine-tuning')]


## 6. Comparing TF-IDF vs. BM25: where does the correct chunk rank?

Rather than eyeballing raw scores, look at the rank of the *actually correct* chunk in each method's results for each query. A rank of 1 means it was the top hit; a chunk missing from the top-`k` altogether shows up as `> k`.

In [8]:
import pandas as pd

def rank_of(target_idx, results):
    for rank, (idx, score, title) in enumerate(results, start=1):
        if idx == target_idx:
            return rank
    return f"> {len(results)}"

rows = []
for query_name, query, target_idx in [
    ("Keyword-style", keyword_query, bm25_chunk_idx),
    ("Paraphrased", paraphrase_query, hallucination_chunk_idx),
]:
    tfidf_results = tfidf_search(query, k=10)
    bm25_results = bm25_search(query, k=10)
    rows.append({
        "query": query_name,
        "tfidf_rank": rank_of(target_idx, tfidf_results),
        "bm25_rank": rank_of(target_idx, bm25_results),
    })

pd.DataFrame(rows)

,query,tfidf_rank,bm25_rank
0,Keyword-style,1,1
1,Paraphrased,4,3


## Takeaways

- Sparse/keyword representations (TF-IDF, BM25) give every chunk one weight per vocabulary word — interpretable, cheap to build, and strong on exact-wording matches.
- Both fail the same way: a query that describes the same idea with different words scores far worse, even though the *meaning* is identical. Neither method has any notion of synonymy (`car` vs. `automobile`, per `rag.pdf`'s own Chapter 2 example).
- BM25 improves on TF-IDF's ranking quality (term-frequency saturation, length normalization) but inherits the same fundamental limitation: it can only match words that are actually present in both the query and the chunk.
- BM25 remains a strong, cheap first-stage retriever in production RAG pipelines — we'll use it again in the hybrid and comparison notebooks later in this series.

**Next up (notebook 4):** **vector embeddings** — dense representations that capture meaning rather than exact words, and directly fix the paraphrase failure demonstrated above.